# 01 · Búsqueda de la configuración

Proyecto Oráculo, ADAG1, Grupo 1. Este notebook documenta cómo se obtuvo la configuración de `entrega.json`.

**Reglas de la búsqueda**

- Se elige solo con las 100 instancias de `busqueda`. La validación se usa al final y no cambia la elección.
- Se usan `evaluar`, `validar`, `CATALOGO`, `espacio` y las trazas. No se abre `oraculo.py` ni se usan sus constantes internas.
- El resultado es la mejor configuración encontrada, no un óptimo global demostrado.

**Antes de empezar:** Entorno de ejecución ▸ Cambiar tipo de entorno de ejecución ▸ T4 GPU. Si Colab se desconecta, vuelve a correr todo: las generaciones guardadas en Drive no se repiten.

## 1 · Instalación

Al terminar, usa **Entorno de ejecución ▸ Reiniciar sesión** y sigue en la sección 2.

In [ ]:
# bitsandbytes y las librerías de nltk/spacy son las que pide el notebook del curso.
!pip install -q -U "bitsandbytes>=0.46.1" transformers accelerate nltk spacy emoji langdetect immutabledict
!python -m spacy download en_core_web_sm -q
!test -d open-instruct || git clone -q https://github.com/allenai/open-instruct

CURSO = "https://raw.githubusercontent.com/JuManoel/ADA-2026-2-GP01/main"
!wget -q --no-cache -O oraculo.py {CURSO}/oraculo.py
!wget -q --no-cache -O ayudas.py {CURSO}/ayudas.py
!wget -q --no-cache -O datos_visibles.json {CURSO}/datos_visibles.json

# Código y entrega del equipo.
!test -d equipo || git clone -q --depth 1 --branch solucion-base https://github.com/samyquito/SYY-Adag1-Oraculo equipo
!cp -r equipo/src . 2>/dev/null || true
!test -f equipo/entrega.json && cp equipo/entrega.json . || true

import io
import urllib.request
import zipfile

NLTK_DATA = "https://raw.githubusercontent.com/nltk/nltk_data/gh-pages/packages"
for carpeta, nombre in [
    ("tokenizers", "punkt"),
    ("tokenizers", "punkt_tab"),
    ("taggers", "averaged_perceptron_tagger"),
    ("taggers", "averaged_perceptron_tagger_eng"),
]:
    with urllib.request.urlopen(f"{NLTK_DATA}/{carpeta}/{nombre}.zip") as resp:
        zipfile.ZipFile(io.BytesIO(resp.read())).extractall(f"/root/nltk_data/{carpeta}")
print("LISTO: reinicia la sesión y sigue en la sección 2")

## 2 · Parámetros

- `HORAS_MAX`: tiempo total de GPU que se quiere gastar en los tres métodos.
- `B`: se calcula en la sección 4 como `min(B_MAXIMO, generaciones posibles en HORAS_MAX / 3)`. Cada método recibe el mismo `B`.
- `MARGEN` y `PISO`: una opción solo se descarta por trazas si la mejor de su ranura ya superó `PISO` y la opción quedó `MARGEN` por debajo. Cerca del piso casi todo marca lo mismo y descartar sería adivinar.

In [ ]:
HORAS_MAX = 3.0
B_MAXIMO = 15_000
MODELO = "qwen17b"
GRUPO = "G01"
SEMANA = 6  # contada desde el 27 de septiembre de 2026, 23:04, con la primera semana = 1
SEMILLA = 0
N0 = 2
MAX_PASADAS = 3
MARGEN = 0.03
PISO = 0.10
TOPE_BACKTRACKING = 2
HOLGURA_COTA = 0.02
SEMILLAS_TEMPERATURA = (1, 2, 3)
DIR_DRIVE = "/content/drive/MyDrive/oraculo_g01"

## 3 · Modelo, datos y oráculo

El caché y los registros quedan en Drive para sobrevivir a una desconexión.

In [ ]:
import json
import os
import time

from google.colab import drive

drive.mount("/content/drive")
os.makedirs(DIR_DRIVE, exist_ok=True)

from ayudas import cargar_datos, cargar_modelo, curva, dividir, entrega
from oraculo import CATALOGO, Oraculo, TEMPERATURAS, espacio
from src.busqueda import (
    RANURAS,
    Consultor,
    ascenso_coordenadas,
    backtracking,
    cota_parcial,
    costo_successive_halving,
    descartar_con_trazas,
    max_candidatos,
    mejores_opciones,
    muestrear_configs,
    successive_halving,
    tamano_espacio,
    validar_entrega,
)

modelo = cargar_modelo(MODELO)
datos = cargar_datos()
busqueda, validacion = dividir(datos)
oraculo = Oraculo(
    modelo, busqueda, validacion,
    cache=f"{DIR_DRIVE}/cache_oraculo.json", lote=16, presupuesto=32_000_000,
)

OPCIONES = {r: list(range(len(CATALOGO[r]))) for r in RANURAS}
N = len(busqueda)
print("|C| sin temperatura:", tamano_espacio(OPCIONES))
print("|C| con temperatura:", tamano_espacio(OPCIONES, temperaturas=TEMPERATURAS))
print("exhaustivo:", tamano_espacio(OPCIONES, temperaturas=TEMPERATURAS) * N, "generaciones")


def evaluar(config, instancias, semilla):
    return oraculo.evaluar(config, instancias, semilla=semilla)

## 4 · Velocidad real y presupuesto `B`

El oráculo no limita las consultas, así que `B` lo fija el equipo con esta medición. El punto de partida sale de una muestra aleatoria con semilla fija.

In [ ]:
inicial = muestrear_configs(OPCIONES, 1, semilla=SEMILLA)[0]
ruta_velocidad = f"{DIR_DRIVE}/velocidad.json"

if os.path.exists(ruta_velocidad):
    velocidad = json.load(open(ruta_velocidad))["generaciones_por_segundo"]
else:
    inicio = time.perf_counter()
    r0 = oraculo.evaluar(inicial, busqueda, semilla=1)
    segundos = time.perf_counter() - inicio
    velocidad = N / segundos
    json.dump({"generaciones_por_segundo": velocidad}, open(ruta_velocidad, "w"))

B = min(B_MAXIMO, int(velocidad * 3600 * HORAS_MAX / 3))
print(f"velocidad: {velocidad:.2f} generaciones/s")
print(f"B por método: {B} generaciones (~{B / velocidad / 3600:.2f} h sin caché)")
print("configuración inicial:", inicial)

## 5 · Ascenso por coordenadas con descarte por trazas

Costo de una pasada sin caché: `Σ k_i · n = 40 · 100 = 4000` generaciones. Las opciones descartadas no se vuelven a medir en las pasadas siguientes.

In [ ]:
def filtro_trazas(ranura, medidas, actual):
    if max(m.precision for m in medidas.values()) < PISO:
        return {}
    return descartar_con_trazas(medidas, actual, MARGEN)


c_coord = Consultor(evaluar, busqueda, presupuesto=B, semilla=1)
r_coord = ascenso_coordenadas(
    c_coord, OPCIONES, inicial, max_pasadas=MAX_PASADAS, n=N, filtro=filtro_trazas,
)
c_coord.guardar(f"{DIR_DRIVE}/registro_coordenadas.json")
print(r_coord.mejor, f"{r_coord.precision:.1%}", r_coord.generaciones, "generaciones")
print("presupuesto agotado:", r_coord.agotado)
for fila in r_coord.detalle["pasadas"]:
    print(fila)

## 6 · Análisis de fallos

Para cada descarte se muestra qué familias de restricciones falló esa opción más veces que la mejor de su ranura, y cuánto ahorró. `violo` es solo la primera restricción incumplida: una traza no prueba que las demás se cumplieran.

In [ ]:
pasadas_hechas = len(r_coord.detalle["pasadas"])
ahorro = 0
for d in r_coord.detalle["descartes"]:
    print(f"pasada {d['pasada']} · {d['ranura']}")
    for opcion, familias in d["opciones"].items():
        principales = list(familias.items())[:3]
        print(f"  opción {opcion}: {CATALOGO[d['ranura']][opcion][:70]!r}")
        print(f"    exceso de fallos: {principales}")
        ahorro += max(0, pasadas_hechas - d["pasada"]) * N
print("generaciones evitadas en las pasadas ya ejecutadas:", ahorro)
if not r_coord.detalle["descartes"]:
    print("No hubo descartes: registrar en el informe que la evidencia no alcanzó PISO y MARGEN.")

resultado_final = oraculo.evaluar(r_coord.mejor, busqueda, semilla=1)
print("\nfamilias que más fallan con la mejor configuración:")
from collections import Counter
for familia, veces in Counter(t["violo"] for t in resultado_final.trazas).most_common(8):
    print(f"  {veces:3}  {familia}")
for traza in resultado_final.trazas[:3]:
    print("-" * 60)
    print("violó:", traza["violo"])
    print(traza["salida"][:300])

**Interpretación (completar después de correr):** qué alternativas se descartaron, por qué familia de fallos y cómo cambió eso la búsqueda.

## 7 · Temperatura (variante)

Con temperatura mayor que 0 el resultado cambia con la semilla, así que se promedian varias. Solo se cambia de 0.0 si el promedio supera estrictamente al de 0.0.

In [ ]:
resultados_t = {}
registro_t = []
for t in TEMPERATURAS:
    config_t = {**r_coord.mejor, "temperatura": t}
    semillas = (1,) if t == 0.0 else SEMILLAS_TEMPERATURA
    precisiones = []
    for s in semillas:
        c_t = Consultor(evaluar, busqueda, semilla=s)
        precisiones.append(c_t.medir(config_t, N).precision)
        registro_t.extend({**f, "semilla": s} for f in c_t.registro)
    resultados_t[t] = sum(precisiones) / len(precisiones)
    print(f"temperatura {t}: {resultados_t[t]:.1%}  {precisiones}")

json.dump(registro_t, open(f"{DIR_DRIVE}/registro_temperatura.json", "w"), indent=1)
mejor_t = 0.0
for t in TEMPERATURAS:
    if resultados_t[t] > resultados_t[mejor_t] + 1e-9:  # 1e-9 evita empates falsos por redondeo
        mejor_t = t
config_coord = {**r_coord.mejor, "temperatura": mejor_t}
precision_coord = resultados_t[mejor_t]
print("configuración de coordenadas:", config_coord, f"{precision_coord:.1%}")

## 8 · Successive Halving con el mismo `B`

Se usa `n_r = min(n0 · 2^r, 100)`, porque sin tope la búsqueda pediría más instancias de las disponibles. Como medir todo `|C|` no cabe en `B`, se parte de una muestra aleatoria del tamaño que sí cabe. Las configuraciones de esta muestra incluyen las tres temperaturas.

In [ ]:
tamano_sh = max_candidatos(B - N, N0, N)
candidatos = muestrear_configs(OPCIONES, tamano_sh, semilla=SEMILLA, temperaturas=TEMPERATURAS)
print("candidatos:", tamano_sh, "costo previsto:", costo_successive_halving(tamano_sh, N0, N, reutiliza=True))

c_sh = Consultor(evaluar, busqueda, presupuesto=B, semilla=1)
r_sh = successive_halving(c_sh, candidatos, N0, N)
c_sh.guardar(f"{DIR_DRIVE}/registro_halving.json")
print(r_sh.mejor, f"{r_sh.precision:.1%}", r_sh.generaciones, "generaciones")
for ronda in r_sh.detalle["rondas"]:
    print(ronda)

## 9 · Backtracking con poda sobre un subespacio

El espacio completo costaría millones de generaciones, así que se explora el subespacio con las `TOPE_BACKTRACKING` mejores opciones de cada ranura según la sección 5.

La cota mide el prompt sin las ranuras aún no decididas, más `HOLGURA_COTA`. Con un modelo real no está demostrado que sea admisible. Por eso este resultado **no** certifica el óptimo: el algoritmo exacto y su análisis se verifican con el oráculo sintético en `tests/`.

In [ ]:
reducidas = mejores_opciones(c_coord.registro, OPCIONES, RANURAS, TOPE_BACKTRACKING, N)
print("subespacio:", reducidas)

c_bt = Consultor(evaluar, busqueda, presupuesto=B, semilla=1)
r_bt = backtracking(
    c_bt, reducidas, cota_parcial(c_bt, N, HOLGURA_COTA),
    base={"temperatura": 0.0}, n=N,
)
c_bt.guardar(f"{DIR_DRIVE}/registro_backtracking.json")
print(r_bt.mejor, f"{r_bt.precision:.1%}", r_bt.generaciones, "generaciones")
print("nodos, hojas y podas:", r_bt.detalle, "| agotado:", r_bt.agotado)

## 10 · Comparación y elección

Gana la mayor precisión sobre `busqueda`. En un empate se prefiere el orden coordenadas, backtracking, halving, porque coordenadas es el que tiene más evidencia de trazas.

In [ ]:
import matplotlib.pyplot as plt

metodos = [
    ("coordenadas", config_coord, precision_coord, c_coord),
    ("backtracking", r_bt.mejor, r_bt.precision, c_bt),
    ("halving", r_sh.mejor, r_sh.precision, c_sh),
]
for nombre, config, precision, consultor in metodos:
    print(f"{nombre:13} {precision:6.1%}  {consultor.generaciones:6} generaciones  {config}")

for nombre, _, _, consultor in metodos:
    filas = [f for f in consultor.registro if f["n"] == N]
    x, y, mejor = [], [], 0.0
    for fila in filas:
        mejor = max(mejor, fila["precision"])
        x.append(fila["acumuladas"])
        y.append(mejor)
    plt.step(x, y, where="post", label=nombre)
plt.xlabel("generaciones")
plt.ylabel("mejor precisión sobre búsqueda (n = 100)")
plt.legend()
plt.savefig(f"{DIR_DRIVE}/comparacion.png", dpi=150, bbox_inches="tight")
plt.show()

curva([f["precision"] for f in c_coord.registro])

ganador = max(metodos, key=lambda m: round(m[2], 9))
metodo_final, config_final, precision_final, _ = ganador
print("elegida:", metodo_final, config_final, f"{precision_final:.1%}")

## 11 · Validación final

Esta celda no vuelve a elegir: solo mide si la configuración aguanta instancias nuevas. No equivale a la evaluación privada, que además incluye familias de restricciones que no están en estos datos.

In [ ]:
r_val = oraculo.validar(config_final)
print(r_val)
print(f"búsqueda: {precision_final:.1%} · validación: {r_val.precision:.1%} sobre {r_val.n} instancias")

**Interpretación (completar después de correr):** diferencia entre búsqueda y validación, y qué limita la evidencia.

## 12 · Entrega

Genera `entrega.json`, comprueba su formato y lo descarga. Súbelo a la raíz del repositorio junto con este notebook ya ejecutado y los registros de Drive.

In [ ]:
from google.colab import files

config_entrega = {
    **{r: int(config_final[r]) for r in RANURAS},
    "temperatura": float(config_final["temperatura"]),
}
d = entrega(GRUPO, config_entrega, SEMANA)
validar_entrega(json.load(open("entrega.json")))
json.dump(d, open(f"{DIR_DRIVE}/entrega.json", "w"), indent=1)
json.dump(
    {
        "metodo": metodo_final, "precision_busqueda": precision_final,
        "precision_validacion": r_val.precision, "B": B, "velocidad": velocidad,
        "generaciones": {n: c.generaciones for n, _, _, c in metodos},
        "temperaturas": {str(t): p for t, p in resultados_t.items()},
        "descartes": r_coord.detalle["descartes"], "backtracking": r_bt.detalle,
    },
    open(f"{DIR_DRIVE}/resumen.json", "w"), indent=1, default=str,
)
files.download("entrega.json")
print("resultados en", DIR_DRIVE)